In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import statsmodels.api as sm
import itertools 

In [2]:
# Imports from China to EU
trade_data = pd.read_csv('data/trade_data_2018_2025.csv')
# Imports from World to EU (incl. China)
trade_data_world = pd.read_csv('data/trade_data_2018_2025_world.csv')

In [ ]:
# Note: Trade data is available only till 2025-09
# trade_data.tail()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
1297,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,9.794778e+07,False,0.0,False,7.316084e+08,NaN,7.316084e+08,2,False,True
1298,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,1.202756e+08,True,0.0,False,5.002984e+08,NaN,5.002984e+08,6,False,True
1299,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,2.192306e+08,False,0.0,False,5.343282e+08,NaN,5.343282e+08,2,False,True
1300,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,7.686484e+07,True,0.0,False,8.195563e+08,NaN,8.195563e+08,6,False,True
1301,C,M,20250901,2025,9,202509,97,EUR,European Union,M,...,8.777624e+07,True,0.0,False,8.313436e+08,NaN,8.313436e+08,6,False,True


In [6]:
trade_data_world_need = trade_data_world[
    ["refPeriodId", "partnerCode", "partnerDesc", "cmdCode", "netWgt"]
]

trade_data_merged = trade_data.merge(
    trade_data_world_need,
    left_on=["refPeriodId", "cmdCode"],
    right_on=["refPeriodId", "cmdCode"],
    suffixes=(None, "_world"),
)

trade_data_merged["china_share_prod"] = (
    trade_data_merged["netWgt"]/ trade_data_merged["netWgt_world"]
)

In [33]:
# trade_data_world_need.shape, trade_data.shape
trade_data_merged.to_csv("a.csv", index=False)

In [7]:
trade_data_merged.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate,post_2025,partnerCode_world,partnerDesc_world,netWgt_world,china_share_prod
0,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,NaN,1.591580e+08,6,False,True,0,0,World,1.076809e+08,0.280189
1,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,NaN,1.273991e+08,0,False,True,0,0,World,3.800298e+07,0.636136
2,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,NaN,4.813467e+08,0,False,True,0,0,World,1.130161e+08,0.696634
3,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,NaN,2.249019e+08,6,False,True,0,0,World,6.193729e+07,0.584436
4,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,NaN,1.282095e+08,2,False,True,0,0,World,2.632927e+07,0.531337


In [ ]:
# grouped_yr_code = trade_data.groupby(["refYear", "cmdCode", "post_2025"]).agg(
#     {"netWgt": "sum"}
# )

In [8]:
trade_data_merged['post_2025'] = 0
trade_data_merged['post_2025'] = np.where(trade_data_merged['refYear'] == 2025, 1, 0)

This is for checking and not needed since I will not use normalized net-weights, but logged net weights

Because of Natural log's (ln) mathematical property where differences equal approximate percentage changes. That's why it puts everything on the same "percentage scale."

The magic property of natural logs:
For small changes, ln(new value) - ln(old value) ≈ percentage change
Let me prove it with our example:
Refrigerators:

Old: 1,000 kg → New: 800 kg
Actual percentage change: (800-1,000)/1,000 = -0.20 = -20%
Using ln: ln(800) - ln(1,000) = 6.685 - 6.908 = -0.223 ≈ -22.3%

Toys:

Old: 100 kg → New: 80 kg
Actual percentage change: (80-100)/100 = -0.20 = -20%
Using ln: ln(80) - ln(100) = 4.382 - 4.605 = -0.223 ≈ -22.3%


The Key Insight
Both get the same ln difference (-0.223) even though:

One started at 1,000 kg
One started at 100 kg

In [ ]:
# mean_pre_2025 = trade_data_merged[trade_data_merged['post_2025'] == 0]['netWgt'].mean()
# mean_post_2025 = trade_data_merged[trade_data_merged['post_2025'] == 1]['netWgt'].mean()

In [ ]:
# trade_data["normalized_wt"] = np.where(
#     trade_data["post_2025"] == 0,
#     trade_data["netWgt"] / mean_pre_2025,
#     trade_data["netWgt"] / mean_post_2025,
# )

In [ ]:
# Std ~0.78 means most values cluster around 0.4–1.6 (25th–75th percentiles).
# Max=4.24 is an outlier (4× mean), but not extreme like raw trade data (where max can be 1000× min).
# No zeros = no need for +1 hack.
# However, max/min ratio is still ~30 so right-skew persists
# trade_data['normalized_wt'].describe() 

count    1302.000000
mean        1.000000
std         0.783528
min         0.138602
25%         0.413174
50%         0.795835
75%         1.260649
max         4.239638
Name: normalized_wt, dtype: float64

In [10]:
# β without normalization is "% change in imports/netweight due to the shock."
# Better to think in growth rates ("15% extra xxx demand? not absolute tons ("+50,000 tons?
# With levels, β is in raw kg ("+X tons per 10pp Asia exposure")—hard to compare across products (washing machines vs. toys have different scales).
# Logging makes β comparable and intuitive:
#  "10pp more Asia exposure → ~2% drop in imports." 
trade_data_merged['log_netwgt'] =  np.log(trade_data_merged['netWgt'])

In [14]:
# Bare-bones model: just interaction + constant
trade_data_merged["interaction"] = (
    trade_data_merged["china_share_prod"] * trade_data["post_2025"]
)
X_bare = sm.add_constant(trade_data_merged["interaction"])  # Constant = intercept

y = trade_data_merged["log_netwgt"]

model_bare = sm.OLS(y, X_bare).fit()

print(model_bare.summary())  # Look at coef on 'interaction' – your first β!

MissingDataError: exog contains inf or nans

In [21]:
print(trade_data_merged[['log_netwgt', 'interaction']].isin([np.inf, -np.inf]).sum())

log_netwgt     0
interaction    0
dtype: int64


In [26]:
trade_data_merged[trade_data_merged["interaction"].isna()]

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,legacyEstimationFlag,isReported,isAggregate,post_2025,partnerCode_world,partnerDesc_world,netWgt_world,china_share_prod,log_netwgt,interaction
5,C,M,20180101,2018,1,201801,97,EUR,European Union,M,...,6,False,True,0,0,World,NaN,NaN,17.701832,NaN
16,C,M,20180201,2018,2,201802,97,EUR,European Union,M,...,6,False,True,0,0,World,NaN,NaN,18.034208,NaN
18,C,M,20180201,2018,2,201802,97,EUR,European Union,M,...,2,False,True,0,0,World,NaN,NaN,16.898441,NaN
40,C,M,20180301,2018,3,201803,97,EUR,European Union,M,...,6,False,True,0,0,World,NaN,NaN,17.936642,NaN
45,C,M,20180401,2018,4,201804,97,EUR,European Union,M,...,6,False,True,0,0,World,NaN,NaN,17.543540,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1227,C,M,20250401,2025,4,202504,97,EUR,European Union,M,...,6,False,True,1,0,World,NaN,NaN,17.931750,NaN
1232,C,M,20250501,2025,5,202505,97,EUR,European Union,M,...,6,False,True,1,0,World,NaN,NaN,17.198648,NaN
1237,C,M,20250501,2025,5,202505,97,EUR,European Union,M,...,6,False,True,1,0,World,NaN,NaN,18.198150,NaN
1246,C,M,20250601,2025,6,202506,97,EUR,European Union,M,...,6,False,True,1,0,World,NaN,NaN,17.321273,NaN


In [ ]:
# FIXME
model_log = sm.OLS(
    "log_netwgt ~ interaction + controls + C(product) + C(time)", data=trade_data
).fit()

print(model_levels.summary())
print(model_log.summary())

ValueError: unrecognized data structures: <class 'str'> / <class 'NoneType'>